# SSL Universal Training Notebook — Colab Pro+ (L4) + VSCode

**사용법**: Cell 3의 `GITHUB_URL` / `BRANCH` 변수와, Cell 4의 `CONFIG` 변수만 바꿔서 같은 노트북으로 모든 method 학습.

- `configs/mocov3_vits.yaml`   — **★ MoCo v3 + ViT-S/16 (현재 메인)**
- `configs/mocov2_mc_r50.yaml` — Track A (multi-crop MoCo, 비교)
- `configs/vicreg_r50.yaml`    — Track B (VICReg, 1차 끝난 뒤)
- `configs/mocov2_r50.yaml`    — 기존 baseline 재현용

**Drive 디렉토리 (1회 사전 생성)**:
```
내 드라이브/ssl_project/
├── data/      # STL10/CIFAR10 raw
├── outputs/   # checkpoints
├── logs/      # training logs
└── features/  # extracted (N, D) feature npy
```

**Pro+ background execution**: 우상단 메뉴 → 백그라운드 실행 ON. 노트북 닫아도 24h 유지.

**워크플로**: VSCode 편집 → `git push` to GitHub 브랜치 → Colab Cell 3 재실행으로 `git pull`. 코드는 Drive에 안 둠 (GitHub가 single source of truth).

In [1]:
# Cell 1 — GPU 확인
import torch, subprocess
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
if torch.cuda.is_available():
    print('VRAM:', torch.cuda.get_device_properties(0).total_memory/1e9, 'GB')
    print('BF16 OK:', torch.cuda.is_bf16_supported())
print(subprocess.check_output(['nvidia-smi', '-L']).decode())
# L4 22GB + bf16 OK 확인. T4면 disconnect 후 Runtime → Change runtime type → L4 재선택.

GPU: NVIDIA L4
VRAM: 23.65915136 GB
BF16 OK: True
GPU 0: NVIDIA L4 (UUID: GPU-f2b8416a-c83e-dcce-507c-ef0ae1bdfab3)



In [2]:
# Cell 2 — Google Drive mount
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# Cell 3 — GitHub repo clone + install + Drive symlinks
#
# ★ 학습 전에 아래 두 변수만 본인 환경에 맞게 수정.
# ★ Private repo면 GITHUB_TOKEN을 Colab Secrets에 등록해서 사용 (아래 주석 참조).

GITHUB_URL = 'https://github.com/minsungk02/Visual-Intelligence-Learning-SSL-DINOFORCE-TV.git'
BRANCH     = 'main'

# ===== Private repo 인증이 필요한 경우 =====
# from google.colab import userdata
# token = userdata.get('GITHUB_TOKEN')   # Colab Secrets에 GITHUB_TOKEN 등록 후 사용
# GITHUB_URL = GITHUB_URL.replace('https://', f'https://{token}@')
# ===========================================

import os, subprocess
os.chdir('/content')

CLONE_DIR = '/content/ssl_project'
if not os.path.exists(CLONE_DIR):
    # 첫 clone — 지정한 브랜치 바로 받아오기
    subprocess.run(['git', 'clone', '-b', BRANCH, GITHUB_URL, CLONE_DIR], check=True)
os.chdir(CLONE_DIR)

# 매 세션마다 fetch + checkout + pull — 최신 코드 동기화
subprocess.run(['git', 'fetch', 'origin'], check=True)
subprocess.run(['git', 'checkout', BRANCH], check=True)
subprocess.run(['git', 'pull', 'origin', BRANCH], check=True)

# 어느 commit에서 학습하는지 기록 — RESULTS.md에 옮겨 적기
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD']).decode().strip()
print(f'Repo:   {GITHUB_URL}')
print(f'Branch: {BRANCH}')
print(f'Commit: {commit}')

# Editable install
subprocess.run(['pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
subprocess.run(['pip', 'install', '-q', '-e', '.'], check=True)

# Drive symlinks — data/outputs/logs/features는 Drive에 영구 저장 (코드는 GitHub)
DRIVE = '/content/drive/MyDrive/ssl_project'
for sub in ['data', 'outputs', 'logs', 'features']:
    os.makedirs(f'{DRIVE}/{sub}', exist_ok=True)
    local = f'/content/ssl_project/{sub}'
    if os.path.islink(local) or os.path.exists(local):
        subprocess.run(['rm', '-rf', local])
    os.symlink(f'{DRIVE}/{sub}', local)

print('Setup complete.')
subprocess.run(['ls', '-la', '/content/ssl_project'])

Repo:   https://github.com/minsungk02/Visual-Intelligence-Learning-SSL-DINOFORCE-TV.git
Branch: main
Commit: 5dbbbfef58f532d1a66ac7cdaf2a4dcba18cca2a
Setup complete.


CompletedProcess(args=['ls', '-la', '/content/ssl_project'], returncode=0)

In [ ]:
# Cell 4 — 학습 시작
#
# 첫 실행에서는 SANITY_EPOCHS=5로 검증 → 통과하면 SANITY_EPOCHS=None으로 풀학습.

import glob, subprocess, os

# ====================================================================
# 학습할 config 선택 (한 줄만 활성화)
# ====================================================================
CONFIG = 'configs/mocov3_vits.yaml';    OUT_KEY = 'outputs/mocov3_vits_seed42'          # ⭐ MoCo v3 + ViT-S/16 (현재 메인)
# CONFIG = 'configs/mocov2_mc_r34.yaml';  OUT_KEY = 'outputs/mocov2_mc_r34_seed42'        # R34 multi-crop (이전 메인)
# CONFIG = 'configs/mocov2_mc_r18.yaml';  OUT_KEY = 'outputs/mocov2_mc_r18_seed42'        # R18 multi-crop (참고)
# CONFIG = 'configs/mocov2_swin_t.yaml'; OUT_KEY = 'outputs/mocov2_swin_t_seed42'         # Swin-T 2-view
# CONFIG = 'configs/mocov2_mc_r50.yaml';  OUT_KEY = 'outputs/mocov2_mc_r50_seed42'        # R50 multi-crop (보류, 시간 초과)
# CONFIG = 'configs/vicreg_r50.yaml';     OUT_KEY = 'outputs/vicreg_r50_seed42'           # VICReg (stretch)
# CONFIG = 'configs/mocov2_r50.yaml';     OUT_KEY = 'outputs/mocov2_r50_seed42'           # baseline 재현

SANITY_EPOCHS = 5   # 첫 실행: 5. 풀학습 진입 시: None

# 자동 resume — Drive에 마지막 ckpt가 있으면 이어서 시작
ckpts = sorted(glob.glob(f'/content/ssl_project/{OUT_KEY}/ckpt_ep*.pth'),
               key=lambda p: int(p.rsplit('_ep', 1)[1].split('.')[0]))
resume_args = ['--resume', ckpts[-1]] if ckpts else []
print('Resume:', ckpts[-1] if ckpts else 'None (fresh start)')

extra = []
if SANITY_EPOCHS is not None:
    extra = ['--epochs', str(SANITY_EPOCHS)]
    print(f'>>> SANITY MODE — {SANITY_EPOCHS} epochs only')

cmd = ['python', '-u', 'scripts/pretrain.py',
       '--config', CONFIG] + resume_args + extra
print('CMD:', ' '.join(cmd))

# stream stdout (Colab UI에 진행 표시)
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        bufsize=1, text=True)
for line in proc.stdout:
    print(line, end='')
proc.wait()
print('Exit code:', proc.returncode)

In [ ]:
# Cell 5 — Feature 추출 (학습 후 또는 중간 평가용)
import subprocess, os

# 추출할 backbone-only checkpoint — 위 Cell 4의 OUT_KEY / CONFIG와 일치시키기.
# ViT 백본은 build_backbone가 config를 읽어야 구조를 재구성할 수 있으므로 --config 필수.
BACKBONE = '/content/ssl_project/outputs/mocov3_vits_seed42/backbone_ep300.pth'
CONFIG   = '/content/ssl_project/configs/mocov3_vits.yaml'
FEAT_DIR = '/content/drive/MyDrive/ssl_project/features/mocov3_vits_ep300'

# ViT + 고정 LP recipe(lr=0.1, wd=0) 조합엔 차원별 z-score(standardize) 권장.
# ResNet 베이스라인 재현 시엔 'none'으로.
NORMALIZE = 'standardize'   # {'none','l2','standardize'}

os.makedirs(FEAT_DIR, exist_ok=True)
subprocess.run(['python', '-u', 'scripts/extract_features.py',
                '--backbone',   BACKBONE,
                '--config',     CONFIG,
                '--output-dir', FEAT_DIR,
                '--normalize',  NORMALIZE,
                '--batch-size', '256'], check=True)
print('Feature dir:', FEAT_DIR)
subprocess.run(['ls', '-la', FEAT_DIR])


In [ ]:
# Cell 6 — LP 평가 (evaluate.py 고정 recipe — 수정 절대 금지)
import subprocess

FEAT_DIR = '/content/drive/MyDrive/ssl_project/features/mocov3_vits_ep300'
subprocess.run(['python', '-u', 'evaluate.py',
    '--stl10-train-features',   f'{FEAT_DIR}/stl10_train_features.npy',
    '--stl10-train-labels',     f'{FEAT_DIR}/stl10_train_labels.npy',
    '--stl10-test-features',    f'{FEAT_DIR}/stl10_test_features.npy',
    '--stl10-test-labels',      f'{FEAT_DIR}/stl10_test_labels.npy',
    '--cifar10-train-features', f'{FEAT_DIR}/cifar10_train_features.npy',
    '--cifar10-train-labels',   f'{FEAT_DIR}/cifar10_train_labels.npy',
    '--cifar10-test-features',  f'{FEAT_DIR}/cifar10_test_features.npy',
    '--cifar10-test-labels',    f'{FEAT_DIR}/cifar10_test_labels.npy',
    '--device', 'cuda'], check=True)

## 운영 팁

- **MoCo v3 + ViT-S/16 (메인) / L4 / batch 1024**: config `backbone.gradient_checkpoint: true`가 activation을 depth배 줄여 24GB에 안전. 만약 그래도 OOM이면 config에서 `batch_size: 512` + `optimizer.lr: 3.0e-4` (lr = 1.5e-4 × batch/256). grad ckpt는 추출 시 자동 무시.
- **L4 22GB / multi-crop(2g+4l) / batch 256** (MoCo v2 트랙): 약 18-20GB 사용. OOM이면 `--batch-size 192` 또는 config의 `gc_mode`를 `"all"`로.
- **bf16 AMP** 활성 — fp16보다 안정, GradScaler 없음. `torch.cuda.is_bf16_supported()`가 True여야.
- **torch.compile**: 첫 epoch에 3-5분 컴파일 오버헤드. resume 시 캐시 무효화 가능 — 가능하면 fresh run을 한 번에.
- **save_every=5 + 자동 resume**: 세션 끊겨도 ≤ 5 epoch 손실. Cell 4가 Drive의 마지막 `ckpt_ep*.pth`를 자동 탐색해 `--resume`. optimizer(AdamW state)·momentum encoder·lr scheduler step까지 복원되어 끊긴 지점에서 그대로 이어짐 (smoke_test_mocov3.py [9]에서 검증).
- **시간 예산 추적**: 첫 10 epoch 평균 시간 × total_epochs로 추정 → 72h 초과 위험 시 epoch 줄여서 중단.
- **코드 업데이트 흐름**: VSCode 편집 → `git push origin minsung` → Colab Cell 3 재실행 (`git pull`이 자동) → Cell 4 재실행.
- **VSCode 운영**: 코드 편집은 VSCode + git push, 실행은 Colab 브라우저. 가장 안정적.